# Lab 07 · Absmax and zero-point

Quantize a vector to int8 two ways and measure reconstruction error.

In [ ]:
import numpy as np

rng = np.random.default_rng(0)
w = rng.normal(size=256).astype(np.float32)

def absmax_q(x, bits=8):
    qmax = 2 ** (bits - 1) - 1
    scale = np.max(np.abs(x)) / qmax
    q = np.clip(np.round(x / scale), -qmax, qmax)
    return q, scale

def dequant_absmax(q, scale):
    return q * scale

def zeropoint_q(x, bits=8):
    qmin, qmax = 0, 2 ** bits - 1
    xmin, xmax = float(x.min()), float(x.max())
    scale = (xmax - xmin) / (qmax - qmin)
    zp = int(np.round(qmin - xmin / scale))
    q = np.clip(np.round(x / scale) + zp, qmin, qmax)
    return q, scale, zp

def dequant_zp(q, scale, zp):
    return (q - zp) * scale

q1, s1 = absmax_q(w)
r1 = dequant_absmax(q1, s1)
q2, s2, zp = zeropoint_q(w)
r2 = dequant_zp(q2, s2, zp)
err1 = float(np.mean((w - r1) ** 2))
err2 = float(np.mean((w - r2) ** 2))
print("absmax mse", err1)
print("zeropoint mse", err2)
assert err1 < 1e-2 and err2 < 1e-2
print("ok")